# AI Newsroom Studio — Pipeline Notebook
10-agent pipeline: HackerNews trends → fact-check → script → video → YouTube Shorts.

**Agents built:** Agent 1 (Trend Hunter) · Agent 2 (Context Researcher) · Agent 3 (Fact Checker)

---
## Setup: Shared State

In [8]:
from typing_extensions import TypedDict
class NewsroomState(TypedDict):
    stories: dict   # per-story (Agents 1-4)
    script:  dict   # NEW top-level key (Agent 5 adds this)
    # script = {
    #   "full_text":    str,   # complete script
    #   "word_count":   int,   # verified count
    #   "est_duration": str,   # "74s"
    #   "sections":     dict,  # parsed labels
    #   "stories_used": list,  # [1, 2, 3] selection ranks
    #   "attempt":      int,   # 1 or 2 (audit trail)
    # }

---
## Agent 1: Trend Fetcher
Fetches top HackerNews stories and scores them by velocity.

**Velocity** = (upvotes + comments×2) / age_hrs — rewards recent engagement.

In [9]:
from agents.agent1 import fetch_trends
import re

def make_story_id(title: str) -> str:
    """Slugify title to a stable dict key. e.g. 'Melody India Italy' → 'melody-india-italy'"""
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

# AGENT 1 NODE
def trend_hunter_node(state: NewsroomState) -> dict:
    trends = fetch_trends()
    stories = {make_story_id(t["title"]): t for t in trends}
    return {"stories": stories}

In [10]:
# Run Agent 1
call = trend_hunter_node(NewsroomState(stories={}))
print(f"Stories fetched: {len(call['stories'])}")
for sid, s in call['stories'].items():
    print(f"  {s['velocity']:>6.1f} vel | {sid}")

Stories fetched: 8
    31.1 vel | backtrack-free-cursive
    28.4 vel | cyberpunk-comics-manga-and-graphic-novels
    26.5 vel | beavis-ultrasound-pnp-isa-sound-card-replica
    22.7 vel | tiny-emulators
    19.3 vel | designing-and-assembling-my-first-pcb
     4.1 vel | ghostlock-a-stack-uaf-that-has-existed-in-all-linux-distributions-for-15-years
     2.2 vel | so-you-want-to-learn-physics-second-edition-2021
     0.2 vel | the-social-physics-of-conversation-communication-patterns-matter


In [11]:
# ── INSPECT after Agent 1 ────────────────────────────────────────────
# Agent 1 fields: title, url, source, category, engagement, velocity
from urllib.parse import urlparse

print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
    31.1          63  technology     mmapped.blog                 Backtrack-Free Cursive
    28.4         228  technology     shellzine.net                Cyberpunk Comics, Manga and Graphic Novels
    26.5          71  technology     github.com                   Beavis Ultrasound PnP ISA Sound Card Replica
    22.7         260  technology     floooh.github.io             Tiny Emulators
    19.3         146  technology     vilkeliskis.com              Designing and assembling my first PCB
     4.1         354  technology     nebusec.ai                   GhostLock, a stack-UAF that has existed in al
     2.2         240  technology     susanrigetti.com             So you want to learn physics (second edition,
     0.2          22  technology     andiroberts.com              The social physics of conversation: Communica

---
## Agent 2: Context Researcher
For each story:
1. Fetches article content (3-tier: trafilatura → Jina → Tavily)
2. Gathers background (DDG news + Wikipedia)
3. Synthesises one tight background paragraph (qwen2.5:3b, content-anchored)

**Key design:**  rejects page chrome before accepting fetched text.
Backgrounds are honest-empty when DDG has no coverage (GitHub/arXiv/new tools — see KNOWN_ISSUES.md).

In [12]:
import ollama, subprocess, time

try:
    ollama.list()
    print("✅ Ollama already running")
except Exception:
    try:
        subprocess.Popen(
            ["ollama", "serve"],
            start_new_session=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        time.sleep(3)
        ollama.list()
        print("✅ Ollama started")
    except Exception:
        print("✗ Run  in a terminal manually")

✅ Ollama started


In [13]:
from agents.agent2 import fetch_url_content, fetch_trend_background

# AGENT 2 NODE
def context_researcher_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with content + background keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        print(f"Agent 2 Starts For : {sid}")
        story["content"]    = fetch_url_content(story["url"])
        story["background"] = fetch_trend_background(story["title"], story["content"])
        print(f"  Agent 2 Ends for: {story['title']}")
        print("=" * 80)
    return {"stories": stories}

In [14]:
# Run Agent 2 (enriches call['stories'] in-place)
call2 = context_researcher_node(call)

Agent 2 Starts For : backtrack-free-cursive
Trafiltura Success ✅  In Loading Content :5068 characters
  [background] gathering snippets for: Backtrack-Free Cursive
  [bg] DDG returned 1 snippets
  [bg] Wikipedia search returns: empty (skipped)
  [background] 1 snippets, synthesizing (content-anchored)...
  [synth] 1 snippets, 229 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 426 chars
Result After Background Syntezing is : The author's experience with cursive writing is influenced by their background in learning the Cyrillic alphabet first, which they find more enjoyable to write than English. This difference in script led them to analyze the problem of backtracking - the need to add strokes to partially written letters - and found that it is rare in Russian but prevalent in English due to its many diacritical marks such as dots and crosses.
  [bg] background: 426 chars
  Agent 2 Ends for: Backtrack-Free Cursive
Agent 2 Starts For : cyberpunk-comics-manga-and-graphic-novels
Trafi

In [15]:
# Run this in notebook after Agent 2 completes
first = next(iter(call2['stories'].values()))
print("Keys after Agent 2:", list(first.keys()))

Keys after Agent 2: ['title', 'url', 'source', 'category', 'engagement', 'velocity', 'content', 'background']


In [16]:
# ── INSPECT after Agent 2 ────────────────────────────────────────────
# Agent 1 fields + Agent 2 new fields: content, background
from urllib.parse import urlparse

# Agent 1 block (unchanged)
print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

# Agent 2 block (new fields only)
print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
    31.1          63  technology     mmapped.blog                 Backtrack-Free Cursive
    28.4         228  technology     shellzine.net                Cyberpunk Comics, Manga and Graphic Novels
    26.5          71  technology     github.com                   Beavis Ultrasound PnP ISA Sound Card Replica
    22.7         260  technology     floooh.github.io             Tiny Emulators
    19.3         146  technology     vilkeliskis.com              Designing and assembling my first PCB
     4.1         354  technology     nebusec.ai                   GhostLock, a stack-UAF that has existed in al
     2.2         240  technology     susanrigetti.com             So you want to learn physics (second edition,
     0.2          22  technology     andiroberts.com              The social physics of co

---
## Agent 3: Fact Checker
Scores each story's credibility on a **-1 to +1 scale**.
Zero is the natural boundary — negative = discard, positive = keep.

| Signal | Base Weight | How |
|--------|-------------|-----|
| `source_score` | 20% | Domain trust map (32 domains, 0.0 to +0.95) |
| `llm_credibility_check` | 60% | gpt-oss-120b → REAL +0.9 / OPINION +0.1 / SPAM -0.7 |
| `cross_verify` | 20% | Exa → DDG → confirms or contradicts via second source |

**Dynamic reweighting** — weights shift when cross-verify fires:
- Contradiction detected → llm 60%→30%, verify 20%→50%
- Confirmation detected  → llm 60%→50%, verify 20%→30%
- Neutral (not found)    → standard weights unchanged

**Key design decisions:**
- `-1 to +1` range — negative range earned by SPAM or credible contradiction
- Threshold = 0.0 — zero is the natural semantic boundary
- SPAM → -0.7 (only genuine negative LLM signal)
- Empty response / Groq failure → 0.0 neutral (never discard on crash)
- Thin content < 500 chars → 0.0 neutral
- Soft discard — story marked `discarded=True`, never deleted (audit trail)
- Cross-verify: Exa (semantic, HN-aware) → DDG fallback → neutral
- Contradiction check uses `compound-mini` (separate quota from 120b)

In [17]:
import time                                    # ← add this line
from agents.agent3 import (
    source_score,
    llm_credibility_check,
    check_credibility,
    cross_verify,
)
print("✅ Agent 3 imported")

✅ Agent 3 imported


In [18]:
# AGENT 3 NODE
def fact_checker_node(state: NewsroomState) -> dict:
    stories = state["stories"]
    for sid, story in stories.items():
        check_credibility(story)
        flag    = "🗑️ DISCARD" if story["discarded"] else "✅ KEEP"
        regime  = story.get("_cred_regime", "?")
        vby     = story.get("verified_by", "NONE")
        contra  = "❌ contradicted" if story.get("contradicted") else ""
        print(f"{story['credibility_score']:+.2f} {flag} "
              f"[{regime}] verified={vby} {contra}| {story['title']}")
        time.sleep(1)
    return {"stories": stories}

In [19]:
# Run Agent 3 on Agent 2's output (no re-run of A1 or A2)
print("── CREDIBILITY RESULTS ─────────────────────────────")
call3 = fact_checker_node(call2)
print("────────────────────────────────────────────────────")

── CREDIBILITY RESULTS ─────────────────────────────
  [llm_cred_check DEBUG] raw='OPINION'
  [llm_cred_check] 'Backtrack-Free Cursive' -> 'OPINION' -> 0.1
  [verify] contradiction check -> 'UNRELATED'
  contradict claim[verify] confirmed by github.com (trust=0.95)
  [cred] regime=confirmation | src=0.00x0.2 + llm=0.10x0.5 + verify=0.80x0.3 = 0.29
+0.29 ✅ KEEP [confirmation] verified=github.com | Backtrack-Free Cursive
  [llm_cred_check DEBUG] raw='REAL'
  [llm_cred_check] 'Cyberpunk Comics, Manga and Graphic Nove' -> 'REAL' -> 0.9
  [verify] neutral -- no credible source found or both tiers failed
  [cred] regime=neutral | src=0.00x0.2 + llm=0.90x0.6 + verify=0.00x0.2 = 0.54
+0.54 ✅ KEEP [neutral] verified=NONE | Cyberpunk Comics, Manga and Graphic Novels
  [llm_cred_check DEBUG] raw='REAL'
  [llm_cred_check] 'Beavis Ultrasound PnP ISA Sound Card Rep' -> 'REAL' -> 0.9
  [verify] neutral -- no credible source found or both tiers failed
  [cred] regime=neutral | src=0.95x0.2 + llm=0.90x

In [20]:
# Run this after Agent 3 completes
first = next(iter(call3['stories'].values()))
print("Keys after Agent 3:", list(first.keys()))

Keys after Agent 3: ['title', 'url', 'source', 'category', 'engagement', 'velocity', 'content', 'background', 'credibility_score', 'verified_by', 'contradicted', 'discarded', '_cred_regime', '_weights_used']


In [21]:
for sid, story in call3["stories"].items():
    print(sid)
    print(story)

backtrack-free-cursive
{'title': 'Backtrack-Free Cursive', 'url': 'https://mmapped.blog/posts/52-backtrack-free-cursive', 'source': 'hackernews', 'category': 'technology', 'engagement': 63, 'velocity': 31.1, 'content': 'Backtrack-free cursive\n✑penmanshipI love writing in cursive, shaping each word in one long stroke. If you grew up learning the Latin alphabet, you likely don’t realize how much joy it sucks out of cursive writing. I noticed only because I learned the Cyrillic alphabet first. I think and write primarily in English, yet Russian feels more enjoyable to write.\nThe crime\nI narrowed the problem to backtracking—the need to add strokes to the letters I’ve partially written. English wants me to dot my i’s and cross my t’s. It has a lot of them, and they like to cluster in a single word. Instead of thinking about what I want to write next, I have to maintain a mental queue of pending strokes.\nBacktracking is rare in Russian. Only й (short i) and э (pronounced like e in end) r

In [22]:
# ── INSPECT after Agent 3 — cumulative (A1 + A2 + A3 fields) ────────
# Keys added: credibility_score, verified_by, contradicted,
#             discarded, _cred_regime, _weights_used
from urllib.parse import urlparse

print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

print()
print('── Agent 3 new fields ──')
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'REGIME':<14} {'WEIGHTS':<12} "
      f"{'VERIFIED_BY':<22} {'SOURCE_DOMAIN':<25} TITLE")
print('-' * 120)
for sid, story in call3['stories'].items():
    flag    = '🗑️' if story.get('discarded') else '✅'
    score   = story.get('credibility_score', 0)
    vel     = story.get('velocity', 0)
    regime  = story.get('_cred_regime', '?')
    weights = story.get('_weights_used', '?')
    vby     = story.get('verified_by', 'NONE')
    contra  = '❌ ' if story.get('contradicted') else ''
    domain  = urlparse(story.get('url','')).netloc.replace('www.','')
    title   = story.get('title','')[:40]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {regime:<14} {weights:<12} "
          f"{contra}{vby:<22} {domain:<25} {title}")


── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
    31.1          63  technology     mmapped.blog                 Backtrack-Free Cursive
    28.4         228  technology     shellzine.net                Cyberpunk Comics, Manga and Graphic Novels
    26.5          71  technology     github.com                   Beavis Ultrasound PnP ISA Sound Card Replica
    22.7         260  technology     floooh.github.io             Tiny Emulators
    19.3         146  technology     vilkeliskis.com              Designing and assembling my first PCB
     4.1         354  technology     nebusec.ai                   GhostLock, a stack-UAF that has existed in al
     2.2         240  technology     susanrigetti.com             So you want to learn physics (second edition,
     0.2          22  technology     andiroberts.com              The social physics of co

In [23]:
from urllib.parse import urlparse
# ── INSPECT: full story details after all 3 agents ──────────────────
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'CONTENT':>8} {'BG':>5} {'REGIME':<14} {'VERIFIED_BY':<19} {'ORIGINAL SOURCE':<25} {'TITLE':<50}")
print("-" * 145)



for sid, story in call3["stories"].items():
    flag    = "🗑️" if story.get("discarded") else "✅"
    score   = story.get("credibility_score", 0)
    vel     = story.get("velocity", 0)
    clen    = len(story.get("content", ""))
    bglen   = len(story.get("background", ""))
    regime  = story.get("_cred_regime", "?")
    vby     = story.get("verified_by", "NONE")
    contra  = "❌" if story.get("contradicted") else ""
    #weights = story.get("_weights_used", "?")
    title   = story.get("title", "")
    source = story.get("url", "")
    source_domain = urlparse(
            source
        ).netloc.replace("www.", "")


    print(    f"{flag}  {score:+.2f}  {vel:>6.1f}  {clen:>6}c  "
            f"{bglen:>4}c  {regime:<14} {contra}{vby:<20}"
            f"{source_domain:<25} {title:<50}")

FLAG SCORE      VEL  CONTENT    BG REGIME         VERIFIED_BY         ORIGINAL SOURCE           TITLE                                             
-------------------------------------------------------------------------------------------------------------------------------------------------
✅  +0.29    31.1    5068c   426c  confirmation   github.com          mmapped.blog              Backtrack-Free Cursive                            
✅  +0.54    28.4    6000c   644c  neutral        NONE                shellzine.net             Cyberpunk Comics, Manga and Graphic Novels        
✅  +0.73    26.5    3038c   767c  neutral        NONE                github.com                Beavis Ultrasound PnP ISA Sound Card Replica      
✅  +0.69    22.7    2967c   346c  confirmation   github.com          floooh.github.io          Tiny Emulators                                    
✅  +0.06    19.3    6000c   737c  neutral        NONE                vilkeliskis.com           Designing and assembling my 

### Save story : till agent 3 (quality data progress saved to disk) so even if some changes comes in agent 1 to agent 3 , agent 4+ code developement will not be affect

In [24]:

# Save to disk — skip re-processing next time
from agent_tools.story_cache import save_stories
save_stories(call3["stories"])

# TO track log of particular function after a hit of particular no. of times pop-up will come


  [cache] saved 8 stories → data/stories_cache.json (16 total in cache)


In [25]:
# importing story cache
# Instead of running A1 → A2 → A3 every time:

from agent_tools.story_cache import load_stories

# Load the saved run (read only — never write again)
cached_stories = load_stories()
print(f"Loaded {len(cached_stories)} stories from cache")


  [cache] loaded 16 stories from data/stories_cache.json
Loaded 16 stories from cache


---
## Agent 4: Editorial

Selects the top stories to cover today from Agent 3's credibility-scored pool.

**Responsibilities:**
1. **Filter** — remove Agent 3 discards (credibility_score < 0.0)
2. **Score** — editorial_score = cred×0.50 + vel_norm×0.30 + bg_norm×0.20
3. **Deduplicate** — phi3.5 clusters titles by topic (one call, no bleed)
4. **Select** — top 3 by editorial_score with topic diversity enforced
5. **Route** — ≥1 story → Agent 5 · 0 stories → end + macOS notification

**First conditional edge in the pipeline** — all previous agents were linear.

In [26]:
# feed directly into Agent 4
from agents.agent4 import editorial_node, route_after_editorial
call4 = editorial_node(call3)
route = route_after_editorial(call4)
print(f"\nRoute: {route}")

AGENT 4: Editorial
  [filter] 8 stories in → 8 eligible (0 discarded by Agent 3)

  [editorial] max velocity this batch: 31.1
  [editorial] Backtrack-Free Cursive                        cred=+0.29 vel=1.00 bg=0.53 → 0.551
  [editorial] Cyberpunk Comics, Manga and Graphic Novels    cred=+0.54 vel=0.91 bg=0.81 → 0.705
  [editorial] Beavis Ultrasound PnP ISA Sound Card Replica  cred=+0.73 vel=0.85 bg=0.96 → 0.812
  [editorial] Tiny Emulators                                cred=+0.69 vel=0.73 bg=0.43 → 0.650
  [editorial] Designing and assembling my first PCB         cred=+0.06 vel=0.62 bg=0.92 → 0.400
  [editorial] GhostLock, a stack-UAF that has existed in al cred=+0.54 vel=0.13 bg=0.64 → 0.438
  [editorial] So you want to learn physics (second edition, cred=+0.06 vel=0.07 bg=0.94 → 0.238
  [editorial] The social physics of conversation: Communica cred=+0.54 vel=0.01 bg=1.00 → 0.472

  [deduplicate] qwen2.5:7b raw output: ```json
[[1], [2], [3], [4], [5], [6], [7], [8]]
```
  [deduplicat

In [27]:
# ── INSPECT after Agent 4 — cumulative (A1 + A2 + A3 + A4 fields) ───
# Keys added: editorial_score, selected, selection_rank,
#             selection_reason, _vel_norm, _bg_norm,
#             _topic_cluster, _is_duplicate
from urllib.parse import urlparse

print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call4['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call4['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

print()
print('── Agent 3 new fields ──')
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'REGIME':<14} {'WEIGHTS':<12} "
      f"{'VERIFIED_BY':<22} {'SOURCE_DOMAIN':<25} TITLE")
print('-' * 120)
for sid, story in call4['stories'].items():
    flag    = '🗑️' if story.get('discarded') else '✅'
    score   = story.get('credibility_score', 0)
    vel     = story.get('velocity', 0)
    regime  = story.get('_cred_regime', '?')
    weights = story.get('_weights_used', '?')
    vby     = story.get('verified_by', 'NONE')
    contra  = '❌ ' if story.get('contradicted') else ''
    domain  = urlparse(story.get('url','')).netloc.replace('www.','')
    title   = story.get('title','')[:40]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {regime:<14} {weights:<12} "
          f"{contra}{vby:<22} {domain:<25} {title}")

print()
print('── Agent 4 new fields ──')
print(f"{'SEL':<5} {'RANK':<5} {'ED_SCORE':<10} {'VEL_N':<7} {'BG_N':<7} "
      f"{'CLUSTER':<9} {'IS_DUPLICATE':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 110)
for sid, story in call4['stories'].items():
    selected  = '✅' if story.get('selected') else '—'
    rank      = story.get('selection_rank', '-')
    ed_score  = story.get('editorial_score', 0)
    vel_n     = story.get('_vel_norm', 0)
    bg_n      = story.get('_bg_norm', 0)
    cluster   = story.get('_topic_cluster', '-')
    is_dup    = '❌' if story.get('_is_duplicate') else '✅'
    domain    = urlparse(story.get('url','')).netloc.replace('www.','')
    title     = story.get('title','')[:40]
    print(f"{selected:<5} #{rank!s:<4} {ed_score:<10.3f} {vel_n:<7.3f} {bg_n:<7.3f} "
          f"{cluster!s:<9} {is_dup:<10} {domain:<28} {title}")

── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
    31.1          63  technology     mmapped.blog                 Backtrack-Free Cursive
    28.4         228  technology     shellzine.net                Cyberpunk Comics, Manga and Graphic Novels
    26.5          71  technology     github.com                   Beavis Ultrasound PnP ISA Sound Card Replica
    22.7         260  technology     floooh.github.io             Tiny Emulators
    19.3         146  technology     vilkeliskis.com              Designing and assembling my first PCB
     4.1         354  technology     nebusec.ai                   GhostLock, a stack-UAF that has existed in al
     2.2         240  technology     susanrigetti.com             So you want to learn physics (second edition,
     0.2          22  technology     andiroberts.com              The social physics of co

In [28]:
# ── Save checkpoint after Agent 5 ────────────────────────────────────
from agent_tools.pipeline_cache import save_checkpoint, list_checkpoints

save_checkpoint(call4, name="till-agent4")
print(f"Available checkpoints: {list_checkpoints()}")

[checkpoint] saved 'till-agent4' → data/checkpoints/till-agent4.json
[checkpoint]   stories: 8
Available checkpoints: ['till-agent4', 'till-agent5', 'till-agent6']


---
## Agent 5: Script Writer
Converts the top 3 selected stories into a 60-90 second YouTube Shorts script.

**Model:** llama-3.3-70b-versatile (Groq) — separate quota from Agent 3
**Format:** HOOK → S1_CONTEXT → S1_CORE → S1_TWIST → S2_HOOK → ... → CTA
**Word target:** 150-225 words (~60-90 seconds at spoken pace)
**Tone calibration:** credibility_score from Agent 3 controls framing confidence

In [29]:
from agents.agent5 import script_writer_node

# Run Agent 5 on Agent 4's output
call5 = script_writer_node(call4)

# Print the full script
print("\n── FULL SCRIPT ──────────────────────────────────────────────────")
print(call5["script"]["full_text"])
print(f"\n── STATS ────────────────────────────────────────────────────────")
print(f"Words:    {call5['script']['word_count']}")
print(f"Duration: ~{call5['script']['est_duration']}")
print(f"Sections: {len(call5['script']['sections'])}")
print(f"Stories:  ranks {call5['script']['stories_used']}")
print(f"Attempts: {call5['script']['attempt']}")

AGENT 5: Script Writer
  [script] 3 selected stories found
  [script]   #1 ed=0.812 | Beavis Ultrasound PnP ISA Sound Card Replica
  [script]   #2 ed=0.705 | Cyberpunk Comics, Manga and Graphic Novels
  [script]   #3 ed=0.650 | Tiny Emulators
  [script] prompt built (8329 chars, 3 stories)
  [script] LLM response: 1609 chars, ~247 words (before enforcement)
  [script] word count after attempt 1: 247 words
  [script] 247 > 225 -> trimming...
  [script] word count after attempt 2: 190 words
  [script] within target (150-225)
  [script] parsed 10 sections: ['HOOK', 'S1_CONTEXT', 'S1_CORE', 'S1_TWIST', 'S2_HOOK', 'S2_CORE', 'S2_TWIST', 'S3_HOOK', 'S3_CORE', 'CTA']

  [script] script complete
  [script]    words:    190
  [script]    duration: ~76s
  [script]    sections: 10
  [script]    attempts: 2
  [script]    stories:  [1, 2, 3]

── FULL SCRIPT ──────────────────────────────────────────────────
HOOK: Eric "Beavis" Schlae released an open-source replica of the Gravis Ultrasound PnP, a l

---
## 🗂️ Cache & Checkpoint Reference

### `story_cache.py` — A1+A2+A3 output
Reload to test Agent 4 or Agent 5 without re-running A1-A3.
```python
from agent_tools.story_cache import load_stories
stories = load_stories()
call4 = editorial_node({"stories": stories})
```
File: `data/stories_cache.json`

### `pipeline_cache.py` — A1+A2+A3+A4+A5 output
Reload to test Agent 6+ without re-running A1-A5.
```python
from agent_tools.pipeline_cache import load_checkpoint
state = load_checkpoint("till-agent5")
call6 = script_qc_node(state)
```
File: `data/checkpoints/till-agent5.json`

| Working on... | Load this | Contains |
|---|---|---|
| Agent 4 or 5 | `load_stories()` | stories only |
| Agent 6+ | `load_checkpoint("till-agent5")` | stories + script |

---

In [30]:
# ── Save checkpoint after Agent 5 ────────────────────────────────────
from agent_tools.pipeline_cache import save_checkpoint, list_checkpoints

save_checkpoint(call5, name="till-agent5")
print(f"Available checkpoints: {list_checkpoints()}")

[checkpoint] saved 'till-agent5' → data/checkpoints/till-agent5.json
[checkpoint]   stories: 8
[checkpoint]   script:  190 words, ~76s
Available checkpoints: ['till-agent4', 'till-agent5', 'till-agent6']


---
## Agent 6: Script QC + Agent 6.1: Voice-Over Generator

Two separate agents with a clean division of labor — text correctness
vs. audio generation reliability.

### Agent 6 — Script QC (text intelligence)
Validates and polishes Agent 5's script until it's genuinely ready to
become audio. Two-stage model split, mirrors Agent 3's design:

| Stage | Model | Job |
|---|---|---|
| JUDGE | `gpt-oss-120b` | Finds problems: word count, AI-voice phrasing, weak transitions, restated twists, CTA format |
| REWRITE | `llama-3.3-70b-versatile` | Fixes ONLY flagged sections — never regenerates the whole script |

**Also handles (pure Python, no LLM cost):**
- Date humanization ("August 23, 2024" → "last year") — uses the real
  current date, since Agent 5 has no clock
- Deterministic `annotated_text` / `tts_ready_text` split — pacing
  markers are derived from sentence structure, never generated by an
  LLM, and **never sent to Kokoro** (Kokoro can't render `[PAUSE]`/
  `[EMPHASIS]` tags — see KNOWN_ISSUES ISSUE-13)

**Control:** APPROVE/REVISE loop, max 2 iterations. Never blocks the
pipeline — approves with a logged warning if issues remain after 2 tries.

### Agent 6.1 — Voice-Over Generator (audio reliability)
Converts Agent 6's QC'd `tts_ready_text` into actual voice-over audio
using **Kokoro** (via `mlx-audio`, Apple Metal GPU acceleration, $0 cost,
fully local — see KNOWN_ISSUES ISSUE-12).

Not in the original 10-agent roadmap — added after Agent 1 itself
surfaced the Kokoro HN story, Agent 3 confirmed it, and Agent 4 selected
it for scripting. Kept **separate** from Agent 6 because text QC and
audio generation are genuinely different concerns with different
failure modes — Agent 6.1 could later swap TTS backends without
touching Agent 6 at all.

**Reliability guarantee, not just "did the command exit 0":**
generates audio, then verifies actual duration roughly matches the
word-count estimate (±50%) — catches garbled or truncated speech that
a bare success/failure check would miss. Only runs if Agent 6 approved
the script — trusts Agent 6's QC gate completely.

In [31]:
from agent_tools.pipeline_cache import load_checkpoint
from agents.agent6 import script_qc_node

state = load_checkpoint("till-agent5")
call6 = script_qc_node(state)

print("\n── ANNOTATED TEXT ──────────────────────────────────")
print(call6["script"]["annotated_text"])
print("\n── TTS-READY TEXT ──────────────────────────────────")
print(call6["script"]["tts_ready_text"])
print(f"\nApproved: {call6['script']['approved']}")
print(f"Iterations: {call6['script']['iterations']}")
print(f"QC notes: {call6['script']['qc_notes']}")

[checkpoint] loaded 'till-agent5' (saved: 2026-07-13T08:47:25.318682+00:00)
[checkpoint]   stories: 8
[checkpoint]   script:  190 words, ~76s
AGENT 6: Script QC
  [qc] date humanization applied to 10 sections

  [qc] --- iteration 1 ---
  [qc] JUDGE empty content, finish_reason=length
  [qc] JUDGE empty/failed on gpt-oss-120b -> falling back to local qwen2.5:7b
  [qc] JUDGE raw: FLAGGED_SECTIONS: S1_TWIST, S2_TWIST
FLAGGED_REASONS: reveals a known risk|restates the core fact
CTA_CATEGORY: A
CTA_OK: YES
  [qc] word_count_ok=True flagged=['S1_TWIST', 'S2_TWIST'] cta_ok=True (category A)
  [qc] rewrote S1_TWIST
  [qc] rewrote S2_TWIST

  [qc] --- iteration 2 ---
  [qc] JUDGE empty content, finish_reason=length
  [qc] JUDGE empty/failed on gpt-oss-120b -> falling back to local qwen2.5:7b
  [qc] JUDGE raw: FLAGGED_SECTIONS: S1_CORE, S2_TWIST, S3_CORE
FLAGGED_REASONS: contains marketing phrases like "requires" | restate the core fact | not under 10 words 
  [qc] word_count_ok=True flagged=['

In [32]:
from agent_tools.pipeline_cache import save_checkpoint
save_checkpoint(call6,'till-agent6')

[checkpoint] saved 'till-agent6' → data/checkpoints/till-agent6.json
[checkpoint]   stories: 8
[checkpoint]   script:  201 words, ~80s


'data/checkpoints/till-agent6.json'

In [33]:

from agents.agent6_1 import voice_over_node
from agent_tools.pipeline_cache import load_checkpoint

pt=load_checkpoint('till-agent6')

call6_1 = voice_over_node(pt)

print(f"Audio generated: {call6_1['script']['audio_generated']}")
print(f"Audio files stitched: {call6_1['script']['audio_chunks']}")
print(f"Duration: {call6_1['script']['audio_duration']}s")
print(f"Verified: {call6_1['script']['duration_verified']}")

[checkpoint] loaded 'till-agent6' (saved: 2026-07-13T08:47:58.342991+00:00)
[checkpoint]   stories: 8
[checkpoint]   script:  201 words, ~80s
AGENT 6.1: Voice-Over Generator (Kokoro)
  [voiceover] generating audio (201 words total)...
  [voiceover] split script into 7 TTS-safe text chunk(s) (max 40 words each)
  [voiceover] generating chunk 1/7 (40 words)...
  [voiceover] generating chunk 2/7 (25 words)...
  [voiceover] generating chunk 3/7 (31 words)...
  [voiceover] generating chunk 4/7 (33 words)...
  [voiceover] generating chunk 5/7 (24 words)...
  [voiceover] generating chunk 6/7 (40 words)...
  [voiceover] generating chunk 7/7 (8 words)...
  [voiceover] chunk 7: mlx_audio exited cleanly but produced no audio_*.wav file
  [voiceover] chunk 7 text: 'Check them out. Follow for daily tech news'
  [voiceover] chunk 7: skipping after retry failed (partial audio will be generated without this chunk)
  [voiceover] 1 chunk(s) skipped: [7]
  [voiceover] stitching 6 audio files into one con

In [ ]:
# ============================================================
# THROWAWAY TEST CELL -- dynamic video POC from any audio file
# Delete this cell once you're done evaluating the approach.
# ============================================================
import wave, math, time
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import subprocess

# ---- point this at any real pipeline audio file ----
AUDIO_DIR = "data/audio"
AUDIO_FILE = "voiceover_20260713_084836.wav"   # <- change per test
STORY_TITLE = "Beavis Ultrasound PnP ISA Sound Card Replica"  # from state["stories"]
SOURCE_DOMAIN = "github.com/schlae"             # from story["url"]
FPS = 30
OUT_DIR = Path("video_poc_output")
# -----------------------------------------------------------

audio_path = str(Path(AUDIO_DIR) / AUDIO_FILE)
OUT_DIR.mkdir(exist_ok=True)
frames_dir = OUT_DIR / "frames"
frames_dir.mkdir(exist_ok=True)

W, H = 720, 1280
BG_COLOR = (13, 17, 23)
ACCENT_COLOR = (88, 166, 255)
TEXT_COLOR = (201, 209, 217)
DIM_COLOR = (139, 148, 158)


def get_amplitude_envelope(wav_path, fps=30):
    with wave.open(wav_path, "rb") as wf:
        n_channels = wf.getnchannels()
        sample_rate = wf.getframerate()
        n_frames = wf.getnframes()
        raw = wf.readframes(n_frames)
    audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32)
    if n_channels == 2:
        audio = audio.reshape(-1, 2).mean(axis=1)
    duration = n_frames / sample_rate
    n_video_frames = int(duration * fps)
    samples_per_frame = len(audio) / n_video_frames
    envelope = np.zeros(n_video_frames)
    for i in range(n_video_frames):
        start, end = int(i * samples_per_frame), int((i + 1) * samples_per_frame)
        chunk = audio[start:end]
        if len(chunk) > 0:
            envelope[i] = np.sqrt(np.mean(chunk ** 2))
    if envelope.max() > 0:
        envelope = envelope / envelope.max()
    window = max(1, fps // 15)
    envelope = np.convolve(envelope, np.ones(window) / window, mode="same")
    return np.clip(envelope, 0, 1)


def load_font(size, bold=False):
    for c in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold
        else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/System/Library/Fonts/Helvetica.ttc",  # macOS fallback
    ]:
        if Path(c).exists():
            return ImageFont.truetype(c, size)
    return ImageFont.load_default()


def make_background():
    cx, cy = W // 2, H // 2 - 100
    max_dist = math.hypot(W, H) * 0.6
    yy, xx = np.mgrid[0:H, 0:W]
    dist = np.clip(np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2) / max_dist, 0, 1)
    lighter, darker = np.array([22, 27, 34]), np.array(BG_COLOR)
    pixels = np.zeros((H, W, 3), dtype=np.uint8)
    for c in range(3):
        pixels[:, :, c] = (lighter[c] * (1 - dist) + darker[c] * dist).astype(np.uint8)
    return Image.fromarray(pixels, "RGB")


def draw_anchor(draw, cx, cy, amp, base_r=90):
    for i in range(3, 0, -1):
        glow_r = base_r + i * 18 * (0.4 + amp)
        alpha = int(40 * amp / i)
        if alpha > 0:
            draw.ellipse([cx-glow_r, cy-glow_r, cx+glow_r, cy+glow_r], fill=ACCENT_COLOR+(alpha,))
    core_r = base_r * (0.85 + 0.25 * amp)
    draw.ellipse([cx-core_r, cy-core_r, cx+core_r, cy+core_r], fill=ACCENT_COLOR+(255,))
    inner_r = core_r * 0.55
    draw.ellipse([cx-inner_r, cy-inner_r, cx+inner_r, cy+inner_r], outline=BG_COLOR+(255,), width=max(2, int(core_r*0.06)))


def draw_bars(draw, cx, cy, window, bar_w=6, gap=5, max_h=70):
    n = len(window)
    x0 = cx - (n * (bar_w + gap) - gap) // 2
    for i, amp in enumerate(window):
        h = max(4, int(amp * max_h))
        x = x0 + i * (bar_w + gap)
        draw.rounded_rectangle([x, cy-h//2, x+bar_w, cy+h//2], radius=bar_w//2, fill=ACCENT_COLOR+(200,))


def draw_lowerthird(img, source, title, fade=1.0):
    draw = ImageDraw.Draw(img, "RGBA")
    y0, h = H - 200, 120
    a = int(200 * fade)
    draw.rectangle([0, y0, W, y0+h], fill=(13,17,23,a))
    draw.rectangle([0, y0, W, y0+3], fill=ACCENT_COLOR+(int(255*fade),))
    ta = int(255 * fade)
    draw.text((32, y0+20), title, font=load_font(30, True), fill=TEXT_COLOR+(ta,))
    draw.text((32, y0+62), f"Source: {source}", font=load_font(22), fill=ACCENT_COLOR+(ta,))


def draw_branding(draw):
    draw.text((32, 50), "AI NEWSROOM STUDIO", font=load_font(26, True), fill=ACCENT_COLOR+(255,))
    draw.text((32, 84), "autonomous · zero human editing", font=load_font(18), fill=DIM_COLOR+(200,))


_bg_cache = None
def render_frame(i, envelope, title, source):
    global _bg_cache
    amp = float(envelope[i])
    if _bg_cache is None:
        _bg_cache = make_background()
    img = _bg_cache.convert("RGBA").copy()
    draw = ImageDraw.Draw(img, "RGBA")
    draw_branding(draw)
    cx, cy = W // 2, H // 2 - 100
    draw_anchor(draw, cx, cy, amp)
    win_start = max(0, i - 19)
    window = envelope[win_start:i+1]
    if len(window) < 20:
        window = np.pad(window, (20 - len(window), 0))
    draw_bars(draw, cx, cy + 160, window)
    fade = min(1.0, i / 15) if i < 200 else 1.0
    draw_lowerthird(img, source, title, fade)
    return img.convert("RGB")


# ---- run it ----
print(f"Analyzing: {audio_path}")
envelope = get_amplitude_envelope(audio_path, fps=FPS)
print(f"{len(envelope)} frames ({len(envelope)/FPS:.1f}s of video)")

t0 = time.time()
for i in range(len(envelope)):
    render_frame(i, envelope, STORY_TITLE, SOURCE_DOMAIN).save(frames_dir / f"frame_{i:05d}.png")
print(f"Rendered in {time.time()-t0:.1f}s")

out_mp4 = OUT_DIR / f"{Path(AUDIO_FILE).stem}_demo.mp4"
subprocess.run([
    "ffmpeg", "-y", "-framerate", str(FPS),
    "-i", str(frames_dir / "frame_%05d.png"),
    "-i", audio_path,
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", "-shortest",
    "-movflags", "+faststart", str(out_mp4)
], check=True, capture_output=True)
print(f"✅ Video ready: {out_mp4}")